Before running the notebook, add dataset: 
Go to File -> Add data.
Search for https://www.kaggle.com/datasets/grassknoted/asl-alphabet . 
Attach this file, the code will read from it using kaggle input 


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
    #for filename in filenames:
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import tensorflow as tf # for loading data, CNNs, training models
from tensorflow.keras import layers, models, callbacks # for CNN layers like Conv2D, MaxPooling, Dense
import matplotlib.pyplot as plt # plotting graphs
import numpy as np # array manipulation
import os # check folder paths
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import seaborn as sns
from IPython.display import FileLink

In [ ]:
data_dir = "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"

print(os.listdir(data_dir))



In [ ]:
img_size = (64, 64) # all images will be 64x64 pixel size
batch_size = 64 # model will train 64 images at a time 


In [ ]:
# training dataset

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, # points to directory
    labels="inferred", # takes the labels as the title of the file. e.g. the file 'A' will hae label 'A'
    label_mode="categorical", # makes one hot vectors (because we are doing multi-class classification)
    image_size=img_size, # evewry image is reszied to 64x64 pixels
    batch_size=batch_size, # batch of 64 
    shuffle=True, # randomises image order to prevent model bias
    validation_split=0.3, # validation and testing dataset is 30% (used to fine tune the model)
    subset="training", # the other 70% is training
    seed=42 # ensures split is the same -> results can be easily reproduced
)


In [ ]:
class_names = train_ds.class_names
print("Classes:", class_names) # creates class names
print("Number of classes:", len(class_names))

In [ ]:
# validation dataset
val_and_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, # points to directory
    labels="inferred", # takes the labels as the title of the file. e.g. the file 'A' will hae label 'A'
    label_mode="categorical", # makes one hot vectors (because we are doing multi-class classification)
    image_size=img_size, # evewry image is reszied to 64x64 pixels
    batch_size=batch_size, # batch of 64 
    shuffle=True, # randomises image order to prevent model bias
    validation_split=0.3, # validation and testing of the dataset is 30% (used to fine tune the model) 
    subset="validation", # lets tensor flow know which portion we want to load (training or validation)
    seed=42 # ensures split is the same -> results can be easily reproduced
)



In [ ]:
# splits dataset into training, validation and testing
val_batches = len(val_and_test_ds) // 2 # divides validaton dataset into 2 equal parts
val_ds = val_and_test_ds.take(val_batches) # takes the first half 
test_ds = val_and_test_ds.skip(val_batches) # takes the remaining half

print(f'Total Training Images: {len(train_ds) * 64}')
print(f'Total Validation Images: {len(val_ds) * 64}')
print(f'Total Test Images: {len(test_ds) * 64}')

In [ ]:
normalization_layer = layers.Rescaling(1./255) # make all pixels between 0 and 1
AUTOTUNE = tf.data.AUTOTUNE

# normalises imgaes, speed up future eppochs, load data faster during training.
# map applies function to every batch of data in the dataset. x =my bat5ch of images, =labels for those images
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(AUTOTUNE)


In [ ]:
# visualise a grid of the first 25 images
# ensures the dataset has loaded correctly - avoid bugs
# confirms resizing worked(64x64)
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1): # take(1) means give one batch of images (64 images)
    for i in range(25):
        ax = plt.subplot(5, 5, i + 1) # one square in a 5x5 grid 
        plt.imshow(images[i].numpy())
        plt.title(class_names[tf.argmax(labels[i]).numpy()])
        plt.axis("off")


In [ ]:
# buid CNN model 
num_classes = len(class_names)

model = models.Sequential([ ## sequential means the layers go in order

    layers.Conv2D(32, (3,3), activation="relu", input_shape=img_size + (3,)), # find patterns like edges and corners
    layers.MaxPooling2D(2, 2), # reduces the image size by taking the max value in every 2x2 block

    layers.Conv2D(64, (3,3), activation="relu"), # increase filters to 64 -> model learns more detailefd patterns
    layers.MaxPooling2D(2, 2), # reduce image size

    layers.Conv2D(128, (3,3), activation="relu"), # doubles filter -> learns complex features
    layers.MaxPooling2D(2, 2), # reduce image size

    layers.Flatten(), # converts 3D feature map to 1D vector

    layers.Dense(128, activation="relu"),# fully connected layer with 128 neurons (tries to understand which gesture the feature is)
    layers.Dropout(0.4), # drops 40% of neurons to prevent overfitting

    layers.Dense(num_classes, activation="softmax") # final output layer
])



In [ ]:
model.summary() #models architecture


In [ ]:
# Compile the model (tells TensorFlow HOW the model should learn)
model.compile(
    optimizer="adam", # tells model how to update the weights
    loss="categorical_crossentropy", # measures how wrong the model is
    metrics=["accuracy"] # shows how accurate the model is and to know if its improving
)


In [ ]:
# early stopping monitors the validation loss and stops trainign when no improvement
early_stopping = callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=3, # waits 3 epochs to restore the weight of lowest val loss
    restore_best_weights = True
)

In [ ]:
# train the CNN
# double check this 
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[early_stopping])

In [ ]:
# Plot accuracy & loss graphs
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs = range(len(acc))

plt.figure(figsize=(12, 5))
plt.suptitle('CNN Training and Validation Metrics', fontsize = 16)
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label="Train Acc")
plt.plot(epochs, val_acc, label="Val Acc")
plt.legend()
plt.title("Accuracy")
plt.xlabel("Epochs")

plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label="Train Loss")
plt.plot(epochs, val_loss, label="Val Loss")
plt.legend()
plt.title("Loss")
plt.xlabel("Epochs")

plt.show()


In [ ]:
# evaluates the trained model on test dataset
test_class_names = class_names
# Collect all labels and predictions 
y_true = []
y_predictions = []


for images, labels in test_ds:

    # generates predictons for the current batch of images
    predictions = model.predict(images, verbose = 0) # verbose = 0 to remove the progress bar
    
    # converts the predicted probabilities to integers
    y_predictions.extend(np.argmax(predictions, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1 ))

# calculates the overall metrics
accuracy = accuracy_score(y_true, y_predictions)
per_acc = accuracy * 100
# marco F1 score to treat the average of all the classes equally
f1 = f1_score(y_true, y_predictions, average='macro' ) 
cm = confusion_matrix(y_true, y_predictions )

print('Final metircs on Test Dataset')
print(f'Test accuracy: {per_acc:.4f}')
print(f'F1 Score: {f1:.4f}')

In [ ]:
# prints the classfication report
print('Classification Report:')
print(classification_report(y_true,y_predictions , target_names = test_class_names))


In [ ]:

# Plots Confusion matrix 

plt.figure(figsize=(16,16))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues", xticklabels=test_class_names, yticklabels=test_class_names)
plt.xlabel("Predicted", fontsize = 12)
plt.ylabel("True", fontsize = 12)
plt.title("CNN ASL Alphabet Confusion Matrix", fontsize = 16)

plt.show()

In [ ]:
# calculate per class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)

for cls, acc in zip(class_names, per_class_acc):
    print(f"{cls:10s}: {acc*100:.2f}%")

In [ ]:
model.save('asl_cnn_model.keras')
print("CNN Model saved!")

In [ ]:
# plot the prediction on the test batch

# Take one batch and predict
images, labels = next(iter(test_ds))

# predict 
preds_probs = model.predict(images, verbose=0)
preds = np.argmax(preds_probs, axis=1)

# plot
plt.figure(figsize=(12, 6))
plt.suptitle('CNN Visual Sample Predictions on Test Data')
for i in range(6):
    ax = plt.subplot(2, 3, i + 1)
    
    img = images[i].numpy().astype("uint8")
    plt.imshow(img)
    

    true_label_idx = np.argmax(labels[i]) if len(labels.shape) > 1 else labels[i]
    pred_label_idx = preds[i]
    
    plt.title(f"Pred: {class_names[pred_label_idx]}\nTrue: {class_names[true_label_idx]}")
    plt.axis("off")

plt.tight_layout()

plt.show()

### Notes on this model:

- **Epoch 1:** The model starts with low accuracy (5.6%) and high loss (3.37%) -> models learning still ofc.
- **Epochs 2-5:** very quick improvement, reaches 97-99% accuracy and 0.001-0.005 val loss. Recognises ASL well.
- **Epochs 6-10:** Training accuracy stabilises around 99% with very low loss. Validation accuracy also stays near 99%.
- **Epochs 11-15:** The model converged fast-> high accuracy (over 99% on both training and val sets)
- No overfitting.